# Milestone 3 — Erster Agent

## Tool-Wrapper: Unsere Chroma-Suche als LangChain-Tool

**Konzept:** Ein LangChain-Tool ist im Kern eine ganz normale Python-Funktion — mit zwei zusätzlichen Dingen, die der Agent braucht, um sie sinnvoll nutzen zu können:

1. Eine **Beschreibung** (meist als Docstring), die dem LLM erklärt, _wofür_ die Funktion gut ist
2. Eine **klare Signatur** (Eingabe-/Ausgabetyp), damit der Agent weiß, was er reinstecken muss

Wir bauen das in zwei Schichten: zuerst die **reine Such-Logik** (Chroma abfragen), dann packen wir sie in einen **LangChain-Tool-Wrapper**, damit der Agent sie "versteht".


In [2]:
%pip install langchain langchain-openai

  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_protocol-0.0.18-py3-none-any.whl.metadata (2.4 kB)
  Using cached uuid_utils-0.17.0-cp313-cp313-macosx_10_12_x86_64.macosx_11_0_arm64.macosx_10_12_universal2.whl.metadata (6.4 kB)
  Using cached jsonpointer-3.1.1-py3-none-any.whl.metadata (2.4 kB)
  Using cached xxhash-3.8.1-cp313-cp313-macosx_11_0_arm64.whl.metadata (15 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
  Using cached zstandard-0.25.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (3.3 kB)
  Using cached tiktoken-0.13.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (6.7 kB)
  Using cached regex-2026.7.19-cp313-cp313-macosx_11_0_arm64.whl.metadata (40 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 202.0 kB/s  0:00:02.4 kB/s eta 0:00:01
Using cached jsonpatch-1.33-py2.py3-none-any.whl (12 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 734.2/734.2 kB 182.4 kB/s  0:00:03202.3 kB/s eta 0:00:

In [3]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import chromadb
from langchain.tools import tool

load_dotenv(dotenv_path="../.env")
client = OpenAI()

# Dieselbe Chroma-DB von M2 wieder öffnen -- kein erneutes Embedden nötig
chroma_client = chromadb.PersistentClient(path="../data/chroma_db")
collection = chroma_client.get_or_create_collection(name="health_fitness_videos")

print("Chunks in der Collection:", collection.count())

Chunks in der Collection: 194


## Die reine Such-Logik

Erst schreiben wir die Funktion, die Chroma tatsächlich abfragt — noch ohne Tool-Wrapper, nur um zu sehen, dass die Suche an sich funktioniert.


In [4]:
def embed_query(text: str) -> list:
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=[text],
    )
    return response.data[0].embedding


def search_video(query: str, n_results: int = 3) -> list:
    query_embedding = embed_query(query)

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
    )

    # Chroma gibt verschachtelte Listen zurück (eine Ebene pro Query) -- wir haben nur eine Query, also [0]
    matches = []
    for doc, metadata in zip(results["documents"][0], results["metadatas"][0]):
        matches.append({
            "text": doc,
            "start": metadata["start"],
            "end": metadata["end"],
        })
    return matches


# Schneller Test, noch ohne Agent
test_results = search_video("What foods are good for brain health?")
for r in test_results:
    print(f"[{r['start']:.1f}s - {r['end']:.1f}s] {r['text'][:150]}...\n")

[562.2s - 593.3s] Some of the most frequent questions I get are about food and the brain. Everybody seems to want to know what they should eat and what they shouldn't e...

[2186.0s - 2219.0s] I eat a fairly limited amount of meat. I don't restrict it, and I do eat meat, but I don't actively seek out creatine in my diet. Rather, I use supple...

[2940.5s - 2970.5s] provided they are taken at reasonable levels. But everything in this list is directed towards answering the question, what can I eat, what can I inges...



## Als LangChain-Tool verpacken

Jetzt verpacken wir `search_video()` in ein echtes LangChain-Tool -- mit dem `@tool`-Decorator. Die **Docstring-Beschreibung ist entscheidend**: Der Agent liest sie, um zu verstehen, wann er dieses Tool einsetzen sollte.


In [5]:
from langchain.tools import tool

@tool
def search_video_tool(query: str) -> str:
    """Durchsucht das Transcript des Videos nach Informationen zu einer bestimmten Frage oder einem Thema.
    Gib eine natürlichsprachliche Frage oder ein Stichwort ein, z.B. 'brain health foods' oder 'creatine supplements'.
    Gibt relevante Textausschnitte mit Zeitstempeln zurück."""

    results = search_video(query, n_results=3)

    # Der Agent bekommt am Ende nur TEXT zurück (kein Python-Objekt) -- also formatieren wir es lesbar
    formatted = ""
    for r in results:
        formatted += f"[{r['start']:.1f}s - {r['end']:.1f}s]: {r['text']}\n\n"

    return formatted


# Kurzer Test: das Tool direkt aufrufen (noch ohne Agent), um zu prüfen, ob es sauber formatiert
print(search_video_tool.invoke("creatine supplements"))

[2028.6s - 2061.2s]: The first author is Roschel, R-O-S-C-H-E-L. We will provide a link to this study, rather, this review, excuse me, in the caption. This was published just very recently in 2021. And one thing to make clear, is that creatine supplementation has been shown to be especially useful for people that are not consuming any meat or other sources of foods that are rich in creatine. What is the threshold level of creatine to supplement in order to get the cognitive benefit? It appears to be at least five grams per day.

[2061.2s - 2093.6s]: Now, the most typical form of creatine is so-called creatine monohydrate. There are other forms of creatine as well, some of which are thought to not draw as much water into non-muscle tissues, and for some people, that's attractive to them. They don't want water sitting below their skin, et cetera. I should emphasize, that the responses to creatine in that sense can differ. Some people get a little bit of water retention. Some people exper

## Den Agent bauen

Jetzt verbinden wir das LLM mit unserem Tool. Wir nutzen LangChains `create_react_agent` -- das implementiert das **ReAct-Pattern** (Reasoning + Acting), das ihr letzte Woche in der Theorie hattet: Der Agent "denkt nach" (Reasoning), entscheidet dann, ob er ein Tool braucht (Acting), sieht sich das Ergebnis an, und wiederholt das bei Bedarf, bevor er antwortet.


In [7]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

# Das "Gehirn" des Agenten -- gpt-4o-mini reicht für unseren MVP, günstig und schnell
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Der Agent bekommt eine Liste von Tools -- aktuell nur unser eines Retrieval-Tool
tools = [search_video_tool]

agent = create_agent(llm, tools)

print("Agent erstellt mit", len(tools), "Tool(s)")

Agent erstellt mit 1 Tool(s)


## Den Agent testen

Wir stellen eine Frage und schauen zu, ob der Agent selbstständig entscheidet, das Retrieval-Tool zu nutzen.


In [8]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "What foods are good for brain health according to this video?"}]
})

# Die finale Antwort ist die letzte Nachricht in der Liste
final_message = response["messages"][-1]
print(final_message.content)

According to the video, some foods that are beneficial for brain health include:

1. **Blueberries and other dark berries** - These are often highlighted in discussions about foods that improve brain function.
2. **Healthy foods in general** - The video emphasizes the importance of consuming a variety of healthy foods to support brain health both in the immediate and long-term.

The speaker also mentions that there is substantial data from peer-reviewed studies indicating the positive impact of certain foods on brain function, memory, and cognition.
